# 🧠 NLP Sentiment Analysis: Multi-Class Product Review Classifier
> **Author:** [Your Name]  
> **Domain:** Natural Language Processing | Machine Learning  
> **Tools:** Python, Scikit-learn, TF-IDF, Logistic Regression, SVM, Random Forest

---

## 🎯 Problem Statement
E-commerce platforms receive millions of product reviews daily. Manually reading and categorizing them is impossible at scale. This project builds an **automated sentiment analysis system** that classifies reviews into **Positive, Negative, or Neutral** — enabling businesses to:
- Monitor brand reputation in real-time
- Identify product quality issues quickly
- Prioritize customer feedback responses
- Generate automated review insights dashboards

## 📐 Project Architecture
```
Raw Reviews → Text Preprocessing → Feature Extraction (TF-IDF) → ML Models → Evaluation → Deployment-Ready API
```

## 📊 Dataset
- **Source:** Custom curated dataset (1,500 product reviews)
- **Classes:** Positive (600), Negative (600), Neutral (300)
- **Features:** review_text, rating, product_category, sentiment
- **Categories:** Electronics, Books, Clothing, Home & Kitchen, Sports, Toys, Beauty, Automotive

## 1. 📦 Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ All libraries imported successfully')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## 2. 📂 Data Loading & Initial Exploration

In [ ]:
df = pd.read_csv('../data/product_reviews.csv')
print('Dataset Shape:', df.shape)
print('\nColumn Types:')
print(df.dtypes)
print('\nFirst 5 Rows:')
df.head()

In [ ]:
print('=== Dataset Overview ===')
print(f'Total Reviews   : {len(df)}')
print(f'Missing Values  : {df.isnull().sum().sum()}')
print(f'Duplicate Rows  : {df.duplicated().sum()}')
print('\nSentiment Distribution:')
print(df['sentiment'].value_counts())
print('\nProduct Categories:')
print(df['product_category'].value_counts())

## 3. 📊 Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis – Product Reviews', fontsize=16, fontweight='bold')

colors = ['#2ECC71', '#E74C3C', '#3498DB']

# Plot 1: Sentiment Distribution
counts = df['sentiment'].value_counts()
axes[0,0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0,0].set_title('Sentiment Class Distribution', fontweight='bold')
axes[0,0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0,0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Plot 2: Rating Distribution
df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], color='#9B59B6')
axes[0,1].set_title('Star Rating Distribution', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=0)

# Plot 3: Review Length
df['review_length'] = df['review_text'].apply(lambda x: len(str(x).split()))
for sent, color in zip(['positive','negative','neutral'], colors):
    axes[1,0].hist(df[df['sentiment']==sent]['review_length'], bins=15, alpha=0.6, label=sent, color=color)
axes[1,0].set_title('Review Length by Sentiment', fontweight='bold')
axes[1,0].set_xlabel('Word Count'); axes[1,0].legend()

# Plot 4: Category Breakdown
cat_sent = df.groupby(['product_category','sentiment']).size().unstack(fill_value=0)
cat_sent.plot(kind='bar', ax=axes[1,1], color=colors, edgecolor='white')
axes[1,1].set_title('Sentiment by Category', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 EDA plots saved')

## 4. 🛠️ Text Preprocessing

**Why preprocessing matters:** Raw text contains noise (punctuation, uppercase, special chars) that hurts model performance. Clean text = better features = better predictions.

**Steps applied:**
1. Lowercase conversion (normalize casing)
2. Remove punctuation & special characters
3. Remove extra whitespace
4. Strip leading/trailing spaces

In [ ]:
def preprocess_text(text):
    """Clean and normalize review text for NLP processing."""
    text = str(text).lower()                        # 1. Lowercase
    text = re.sub(r'[^a-z\s]', '', text)            # 2. Remove non-alphabetic
    text = re.sub(r'\s+', ' ', text).strip()         # 3. Remove extra whitespace
    return text

# Apply preprocessing
df['clean_text'] = df['review_text'].apply(preprocess_text)

# Show before/after
print('=== Preprocessing Example ===')
print('BEFORE:', df['review_text'].iloc[0])
print('AFTER :', df['clean_text'].iloc[0])
print(f'\n✅ Preprocessed {len(df)} reviews')

## 5. 🔢 Feature Engineering: TF-IDF Vectorization

**TF-IDF (Term Frequency–Inverse Document Frequency):** A numerical statistic reflecting how important a word is to a document in the corpus.

- **TF** = how often word appears in document
- **IDF** = how rare/important the word is across all documents
- **Bigrams (n-gram=2):** captures phrases like "not good", "highly recommend"

In [ ]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set   : {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test set       : {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTrain class dist:\n{y_train.value_counts()}')
print(f'\nTest class dist:\n{y_test.value_counts()}')

## 6. 🤖 Model Training – 5 Algorithms Compared

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
    ]),
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True)),
        ('clf', LinearSVC(max_iter=2000, C=0.5, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000)),
        ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42))
    ]),
}

results = {}
print(f'{"Model":<25} | {"Accuracy":>10} | {"F1-Score":>10}')
print('-' * 52)

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    results[name] = {'accuracy': acc, 'f1': f1, 'pipeline': pipeline, 'y_pred': y_pred}
    print(f'{name:<25} | {acc:>10.4f} | {f1:>10.4f}')

## 7. 📈 Model Evaluation & Visualization

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best = results[best_name]
print(f'🏆 Best Model: {best_name}')
print('\n=== Detailed Classification Report ===')
print(classification_report(y_test, best['y_pred']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Model Evaluation – {best_name}', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, best['y_pred'], labels=['positive','neutral','negative'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['positive','neutral','negative'],
            yticklabels=['positive','neutral','negative'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

# Model Comparison
model_names = list(results.keys())
accs = [results[m]['accuracy'] for m in model_names]
f1s  = [results[m]['f1'] for m in model_names]
x = np.arange(len(model_names))
axes[1].bar(x-0.2, accs, 0.35, label='Accuracy', color='#3498DB', alpha=0.85)
axes[1].bar(x+0.2, f1s,  0.35, label='F1-Score',  color='#E74C3C', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=25, ha='right', fontsize=9)
axes[1].set_ylim(0, 1.1); axes[1].legend()
axes[1].set_title('Model Comparison')

plt.tight_layout()
plt.savefig('../outputs/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 🔍 Feature Importance – What Makes a Review Positive or Negative?

In [ ]:
lr_pipeline  = results['Logistic Regression']['pipeline']
vectorizer   = lr_pipeline.named_steps['tfidf']
clf          = lr_pipeline.named_steps['clf']
feature_names = vectorizer.get_feature_names_out()
classes = clf.classes_

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 20 Predictive Features per Sentiment Class', fontsize=13, fontweight='bold')

palette = {'positive': '#2ECC71', 'neutral': '#3498DB', 'negative': '#E74C3C'}
for ax, cls in zip(axes, ['positive', 'neutral', 'negative']):
    idx = list(classes).index(cls)
    coefs = clf.coef_[idx]
    top20_idx   = np.argsort(coefs)[-20:]
    top20_words = [feature_names[i] for i in top20_idx]
    top20_vals  = coefs[top20_idx]
    ax.barh(top20_words, top20_vals, color=palette[cls], alpha=0.85)
    ax.set_title(f'{cls.title()} Sentiment', fontweight='bold')
    ax.set_xlabel('Coefficient Weight')

plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance charts saved')

## 9. 🚀 Real-Time Inference Demo

In [ ]:
def predict_sentiment(review_text, pipeline=lr_pipeline):
    """Predict sentiment for any input review text."""
    cleaned = preprocess_text(review_text)
    prediction = pipeline.predict([cleaned])[0]
    emoji_map = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}
    return prediction, emoji_map[prediction]

# Test with custom reviews
custom_reviews = [
    "This product is absolutely fantastic, best purchase ever!",
    "Terrible quality, stopped working after two days. Very disappointed.",
    "It arrived on time and works okay, nothing special.",
    "Incredible value for money, extremely happy with this!",
    "Complete waste of money, broken out of the box.",
    "Decent product, does what it says on the tin."
]

print('=== Live Sentiment Prediction Demo ===')
print('-' * 70)
for review in custom_reviews:
    pred, emoji = predict_sentiment(review)
    print(f'{emoji} [{pred.upper():8s}] {review[:60]}')
print('\n✅ Model ready for deployment!')

## 10. 💾 Save Best Model for Deployment

In [ ]:
import pickle

best_pipeline = results[best_name]['pipeline']

with open('../outputs/best_model.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

# Verify it loads correctly
with open('../outputs/best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

test_text = 'This is absolutely wonderful, love it'
print(f'Model loaded & tested: "{test_text}"')
print(f'Prediction: {loaded_model.predict([preprocess_text(test_text)])[0]}')
print(f'\n✅ Model saved as best_model.pkl')

## 11. 📋 Project Summary & Key Findings

| Model | Accuracy | F1 Score |
|---|---|---|
| Logistic Regression | ~1.00 | ~1.00 |
| Naive Bayes | ~1.00 | ~1.00 |
| Linear SVM | ~1.00 | ~1.00 |
| Random Forest | ~1.00 | ~1.00 |
| Gradient Boosting | ~1.00 | ~1.00 |

### Key Learnings
1. **TF-IDF with bigrams** captures context phrases (e.g., 'not good', 'highly recommend')
2. **Logistic Regression** is fast, interpretable, and competitive for text classification
3. **Feature importance** reveals meaningful linguistic patterns per sentiment class
4. The **pipeline approach** ensures preprocessing + model is consistent for deployment

### Potential Enhancements
- Fine-tune a **BERT/DistilBERT** model for better contextual understanding
- Add **aspect-based sentiment** (identify what specifically is positive/negative)
- Build a **Flask/FastAPI** REST endpoint for production serving
- Deploy on **Hugging Face Spaces** or **AWS SageMaker**